In [20]:
import pandas as pd

In [21]:
df = pd.read_csv(r"../data/processed/clean_supply_chain.csv")
df.head()

,Product_ID,Warehouse,Order_Date,Shipment_Date,Lead_Time,Demand_Forecast,Inventory_Level,Stockout_Flag,Backorder_Flag,Supplier_ID,Order_Quantity,Shipment_Quantity,Product_Category,Product_Price,Customer_ID,Order_Priority,Revenue
0,8270,A,2023-01-01,2023-01-02,8,839,481,0,0,397,98,40,Clothing,387.89,2014,High,38013.22
1,1860,C,2023-01-01,2023-01-02,8,351,332,0,1,483,52,35,Furniture,891.03,6805,High,46333.56
2,6390,C,2023-01-01,2023-01-02,5,398,156,0,0,160,34,43,Food,933.21,3833,Medium,31729.14
3,6191,A,2023-01-01,2023-01-02,5,972,418,1,0,161,32,45,Electronics,628.57,3640,Medium,20114.24
4,6734,D,2023-01-01,2023-01-02,3,529,428,0,0,325,44,41,Electronics,70.33,9297,Medium,3094.52


In [22]:
#WAREHOUSE
warehouse_perf = (
    df.groupby('Warehouse')
      .agg({
          'Lead_Time':'mean',
          'Revenue':'sum',
          'Stockout_Flag':'mean',
          'Backorder_Flag':'mean'
      })
)
warehouse_perf

,Lead_Time,Revenue,Stockout_Flag,Backorder_Flag
Warehouse,,,,
A,4.977659,57412206.54,0.205541,0.162645
B,4.908725,55855897.16,0.194631,0.158389
C,4.958916,57502720.76,0.202257,0.149887
D,5.034602,59039313.87,0.203287,0.167820


In [23]:
warehouse_perf.sort_values(
    by='Revenue',
    ascending=False
)

,Lead_Time,Revenue,Stockout_Flag,Backorder_Flag
Warehouse,,,,
D,5.034602,59039313.87,0.203287,0.167820
C,4.958916,57502720.76,0.202257,0.149887
A,4.977659,57412206.54,0.205541,0.162645
B,4.908725,55855897.16,0.194631,0.158389


In [24]:
# Warehouse D has the highest revenue.
# Warehouse D has the highest average lead time.
# Warehouse A has the highest stockout rate.

In [25]:
#SUPPLIER
supplier_perf = (
    df.groupby('Supplier_ID')
      .agg({
          'Lead_Time':'mean'
      })
      .sort_values(
          'Lead_Time',
          ascending=False
      )
)


In [26]:
supplier_perf.head(10)

,Lead_Time
Supplier_ID,
263,6.500000
178,6.437500
209,6.388889
265,6.187500
205,6.086957
241,6.080000
449,6.047619
268,6.000000
414,5.972222


In [27]:
supplier_perf.tail(10)

,Lead_Time
Supplier_ID,
429,3.964286
113,3.962963
440,3.866667
220,3.750000
382,3.739130
224,3.666667
343,3.652174
234,3.333333
340,3.200000


In [28]:
#REVENUE ANALYSIS
category_revenue = (
    df.groupby('Product_Category')
      ['Revenue']
      .sum()
      .sort_values(
          ascending=False
      )
)
category_revenue

Product_Category
Electronics    59499850.20
Clothing       59402875.11
Furniture      56949757.22
Food           53957655.80
Name: Revenue, dtype: float64

In [29]:
warehouse_revenue = (
    df.groupby('Warehouse')['Revenue']
      .sum()
      .sort_values(ascending=False)
)
warehouse_revenue

Warehouse
D    59039313.87
C    57502720.76
A    57412206.54
B    55855897.16
Name: Revenue, dtype: float64

In [30]:
#The Most revenue genrates by D warehouse

In [31]:
#Stockout Analysis
stockout_analysis = (
    df.groupby('Warehouse')
      ['Stockout_Flag']
      .mean()
      .sort_values(
          ascending=False
      )
)
stockout_analysis

Warehouse
A    0.205541
D    0.203287
C    0.202257
B    0.194631
Name: Stockout_Flag, dtype: float64

In [32]:
#Backorder Analysis 
backorder_analysis = (
    df.groupby('Product_Category')
      ['Backorder_Flag']
      .mean()
      .sort_values(
          ascending=False
      )
)
backorder_analysis

Product_Category
Furniture      0.166741
Food           0.165602
Clothing       0.154286
Electronics    0.152766
Name: Backorder_Flag, dtype: float64

In [33]:
#Inventory Analysis
df['Inventory_Coverage'] = (
    df['Inventory_Level']
    / df['Demand_Forecast']
)

In [34]:
inventory_analysis = (
    df.groupby('Product_Category')
      ['Inventory_Coverage']
      .mean()
      .sort_values()
)
inventory_analysis

Product_Category
Food           0.615038
Electronics    0.620169
Clothing       0.633653
Furniture      0.646513
Name: Inventory_Coverage, dtype: float64

In [35]:
#Demand vs Supply Gap
df['Demand_Gap'] = (
    df['Demand_Forecast']
    - df['Shipment_Quantity']
)

In [36]:
gap_analysis = (
    df.groupby('Product_Category')
      ['Demand_Gap']
      .mean()
      .sort_values(ascending=False)
)
gap_analysis

Product_Category
Food           505.529197
Electronics    504.514486
Clothing       500.254505
Furniture      497.623947
Name: Demand_Gap, dtype: float64

In [37]:
#Priority Analysis
priority_analysis = (
    df.groupby('Order_Priority')
      .agg({
          'Lead_Time':'mean',
          'Shipment_Quantity':'mean'
      })
)
priority_analysis

,Lead_Time,Shipment_Quantity
Order_Priority,,
High,4.96991,49.934470
Low,4.94437,48.823784
Medium,4.99734,49.470236


In [38]:
import os
from sqlalchemy import create_engine
from dotenv import load_dotenv

load_dotenv()

DB_HOST = os.getenv("DB_HOST")
DB_USER = os.getenv("DB_USER")
DB_PASSWORD = os.getenv("DB_PASSWORD")
DB_NAME = os.getenv("DB_NAME")

In [39]:
engine = create_engine(
    f"mysql+pymysql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}/{DB_NAME}"
)
print("Connected Successfully!")

Connected Successfully!


In [40]:
df.to_sql("raw_data", engine, if_exists="replace", index=False)

9000